# CardioGuard AI — Training on Kaggle (GPU)

Runs the full pipeline: **preprocess → train → export ONNX → clinical test**, using the honest **inter-patient split**.

### Before you run
1. Turn the GPU on: **Settings (right panel) → Accelerator → GPU T4 x2** (or P100).
2. Add your data: **Add Input → Datasets →** your uploaded dataset that contains the project (`preprocess.py`, `train.py`, `split_utils.py`, `export_onnx.py`, `clinical_testing/`, and `data/mitbih/`).
3. Set `DATASET_DIR` in the setup cell below to that dataset's path under `/kaggle/input/...`.

In [ ]:
# 1) Install the extra libraries Kaggle doesn't ship with
!pip install -q wfdb tf2onnx onnx onnxruntime

In [ ]:
# 2) Auto-find the project folder, copy it into the writable dir, and cd in
import os, shutil, glob

# Auto-find the folder that contains the scripts (handles any dataset name/nesting)
DATASET_DIR = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "preprocess.py" in files and "split_utils.py" in files:
        DATASET_DIR = root
        break
assert DATASET_DIR, "Couldn't find preprocess.py under /kaggle/input — is the dataset attached?"
print("Found project at:", DATASET_DIR)

PROJ = "/kaggle/working/proj"
if os.path.exists(PROJ):
    shutil.rmtree(PROJ)
shutil.copytree(DATASET_DIR, PROJ)
os.chdir(PROJ)

print("cwd:", os.getcwd())
print("MIT-BIH records found:", len(glob.glob('data/mitbih/*.dat')), "(expect 48)")

# Confirm the GPU is visible to TensorFlow
import tensorflow as tf
print("TF version:", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 3) Preprocess MIT-BIH -> X.npy, y.npy, groups.npy  (groups enables the inter-patient split)
!python preprocess.py

In [ ]:
# 4) Train on GPU. Watch the '[inter-patient split]' printout: test uses UNSEEN records.
!python train.py --epochs 60 --batch 64

In [ ]:
# 5) Export to ONNX.
# Trained-on-GPU LSTMs bake a cuDNN op (CudnnRNNV3) that onnxruntime can't run,
# so we re-export the SavedModel with the GPU hidden (CUDA_VISIBLE_DEVICES="")
# to get a generic LSTM that converts to a standard ONNX LSTM.
import os, shutil
os.makedirs('ai', exist_ok=True)
SM = '/kaggle/working/proj/saved_model'
if os.path.exists(SM): shutil.rmtree(SM)
if os.path.exists('ai/cardioguard_model.onnx'): os.remove('ai/cardioguard_model.onnx')

export_code = (
    'import tensorflow as tf\n'
    'm = tf.keras.models.load_model("models/best_model.keras")\n'
    'try:\n'
    '    m.export("%s")\n'
    'except AttributeError:\n'
    '    tf.saved_model.save(m, "%s")\n'
) % (SM, SM)
open('_export_sm.py', 'w').write(export_code)

get_ipython().system('CUDA_VISIBLE_DEVICES="" python _export_sm.py')
get_ipython().system('python -m tf2onnx.convert --saved-model {SM} --output ai/cardioguard_model.onnx --opset 13')

import onnxruntime as ort, numpy as np
sess = ort.InferenceSession('ai/cardioguard_model.onnx', providers=['CPUExecutionProvider'])
name = sess.get_inputs()[0].name
out = sess.run(None, {name: np.random.randn(1, 256, 1).astype(np.float32)})
print('ONNX loads and runs — output shape', out[0].shape,
      '| size MB:', round(os.path.getsize('ai/cardioguard_model.onnx')/1e6, 2))

In [ ]:
# 6) Clinical test: honest metrics on the unseen-patient test set + confusion matrix
!python clinical_testing/run_clinical_tests.py

In [ ]:
# 7) Bundle everything to download (Output tab, right panel)
import shutil, os
os.makedirs('/kaggle/working/cardioguard_outputs', exist_ok=True)
for src in ['ai/cardioguard_model.onnx', 'models/best_model.keras', 'models/history.json',
            'models/evaluation.txt', 'models/plots', 'clinical_testing/results']:
    if os.path.exists(src):
        dst = os.path.join('/kaggle/working/cardioguard_outputs', os.path.basename(src.rstrip('/')))
        (shutil.copytree if os.path.isdir(src) else shutil.copy)(src, dst)
shutil.make_archive('/kaggle/working/cardioguard_outputs', 'zip', '/kaggle/working/cardioguard_outputs')
print('Download: cardioguard_outputs.zip  (from the Output/Data panel)')